Import Libraries

In [1]:
import numpy as np
import pandas as pd

Import Data

In [3]:
data=pd.read_csv('/content/emails.csv')
data.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


In [4]:
data.describe()

,spam
count,5728.000000
mean,0.238827
std,0.426404
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,1.000000


In [5]:
list(data)

['text', 'spam']

In [6]:
data.isnull().sum()

,0
text,0
spam,0


Data Processing

In [7]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [8]:
v=set(stopwords.words('english'))
print(v)

{'to', 'why', 'do', 'off', "hasn't", 'out', 'you', 'his', 'what', 'nor', 'which', 'now', 'she', 'he', 'from', 'have', 'y', 'for', 'couldn', 'isn', 'o', 'once', 'down', 'most', "won't", 'd', 're', 'very', 'its', 'on', "you're", "aren't", 'no', 'against', 'these', 'does', "needn't", 'herself', 'during', "that'll", 'we', 'will', 'at', "he's", 'am', 'after', 'a', 'they', 'wasn', 'ours', 'about', 'because', 'yourself', 'mustn', 'and', "i've", "hadn't", "he'd", 'being', 'in', 'too', 'them', 'doesn', "wouldn't", "didn't", 'than', 'below', 'such', 'more', "weren't", "i'll", 'doing', 'were', 'whom', 'i', "i'm", 'our', 'is', 'some', 'are', 'before', "shouldn't", 'hasn', "shan't", 'only', 'aren', 'under', 'was', 'itself', 'it', "haven't", 'himself', 't', 'me', 've', "should've", 'shouldn', 'then', "they'll", 'between', 'wouldn', 'who', 'ma', 'not', 'here', "they're", 'of', 'until', 'ain', "she'll", "we'll", 'so', 'your', 'while', 'both', 'been', 'there', 'yourselves', 'how', "mightn't", "they've"

In [9]:
import re

In [10]:
bag=[]
for i in range(0,5728):
  review=re.sub('[^a-zA-Z]',' ',data['text'][i])
  review=review.lower()
  review=review.split()
  lm=PorterStemmer()
  review=[lm.stem(word) for word in review if not word in set(v)]
  review=' '.join(review)
  bag.append(review)

In [11]:
bag

['subject natur irresist corpor ident lt realli hard recollect compani market full suqgest inform isoverwhelminq good catchi logo stylish statloneri outstand websit make task much easier promis havinq order iogo compani automaticaili becom world ieader isguit ciear without good product effect busi organ practic aim hotat nowaday market promis market effort becom much effect list clear benefit creativ hand made origin logo special done reflect distinct compani imag conveni logo stationeri provid format easi use content manag system letsyou chang websit content even structur prompt see logo draft within three busi day afford market break make gap budget satisfact guarante provid unlimit amount chang extra fee surethat love result collabor look portfolio interest',
 'subject stock trade gunsling fanni merril muzo colza attaind penultim like esmark perspicu rambl segovia group tri slung kansa tanzania ye chameleon continu clothesman libretto chesapeak tight waterway herald hawthorn like ch

Creating the bag of words model

In [12]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=1600)
X=cv.fit_transform(bag).toarray()
y=data['spam']

Splitting the dataset into the training set and test set

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.25, random_state=0)

# Training the Naive Bayes model on the training set

In [14]:
from sklearn.naive_bayes import GaussianNB
classifier=GaussianNB()
classifier.fit(X_train,y_train)

GaussianNB()

Predicting the Test set results

In [15]:
y_pred=classifier.predict(X_test)

Making the confusion matrix

In [16]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm=confusion_matrix(y_test,y_pred)
print(cm)
accuracy_score(y_test,y_pred)

[[1039   50]
 [  23  320]]


0.9490223463687151

To check model prediction

In [30]:
new_review="dr . kaminski called me back telling me that he would like to have you as  an intern student in his research department during the summer . please  write him an email as soon as possible to introduce yourself and letting  him know of your expected starting date and ending date"
new_review=re.sub('[^a-zA-Z]',' ',new_review)
new_review=new_review.lower()
new_review=new_review.split()
ps=PorterStemmer()
all_stopwords=stopwords.words('english')
all_stopwords.remove('not')
new_review=[ps.stem(word) for word in new_review if not word in set(all_stopwords)]
new_review=" ".join(new_review)
new_corpus=[new_review]
new_X_test=cv.transform(new_corpus).toarray()
new_y_pred=classifier.predict(new_X_test)
print(new_y_pred)

[0]


# Training the Random Forest model on the training set

In [31]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

In [32]:
cls=RandomForestClassifier()
n_estimators=[25,50,75,100,125,150,175,200]
default=100
criterion=['gini','entropy']
max_depth=[3,5,10]
parameters={'n_estimators':n_estimators,'criterion':criterion,'max_depth':max_depth}
rfc=GridSearchCV(cls,parameters)
rfc.fit(X_train,y_train)

GridSearchCV(estimator=RandomForestClassifier(),
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [3, 5, 10],
                         'n_estimators': [25, 50, 75, 100, 125, 150, 175, 200]})

In [33]:
rf_pred=rfc.predict(X_test)

In [34]:
confusion_matrix(y_test,rf_pred)

array([[1087,    2],
       [  82,  261]])

In [35]:
accuracy_score(y_test,rf_pred)

0.9413407821229051